# KneeAssist AI V1 — environment and data checks
This notebook uses the prepared MRNet data and saved leakage audit. It does not reinspect ZIP datasets or change the split. Run with the **KneeAssist AI** kernel / project `.venv` interpreter.

The available cohort has study identifiers, not verified patient linkage. The audit checks study separation and exact series duplicates; it cannot establish patient independence without the optional mapping.


In [1]:
from pathlib import Path
import sys, json
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "config.yaml").exists() and (p / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from KneeAssist-AI or its notebooks folder.")
sys.path.insert(0, str(ROOT))
from src.utils import config
cfg = config(ROOT / "config.yaml")
print("Project:", ROOT)
print("Python:", sys.executable)


Project: C:\Users\HP\Desktop\KneeAssist-AI
Python: C:\Users\HP\Desktop\KneeAssist-AI\.venv\Scripts\python.exe


In [2]:
import torch, torchvision, pandas as pd
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Encoder precision:", cfg["model"]["encoder_precision"])
print("Classifier mixed precision:", cfg["training"]["mixed_precision"])


PyTorch: 2.11.0+cu128
torchvision: 0.26.0+cu128
CUDA available: True
GPU: NVIDIA T500
GPU memory (GiB): 4.0
Encoder precision: float32
Classifier mixed precision: True


In [3]:
from src.data.dataset import read_manifest
rows = read_manifest(cfg)
manifest = pd.DataFrame(rows)
manifest["study_id"] = manifest["study_id"].astype(str)
display(manifest.groupby("split").size().rename("Studies").to_frame())
labels = [target["key"] for target in cfg["targets"]]
display(manifest.assign(**{k: manifest[k].astype(int) for k in labels}).groupby("split")[labels].sum().rename_axis("Positive labels"))


,Studies
split,
train,1130
valid,120


,abnormal,acl,meniscus
Positive labels,,,
train,913,208,397
valid,95,54,52


In [4]:
split = json.loads((ROOT / cfg["dataset"]["splits"]).read_text())
parts = {name: set(ids) for name, ids in split.items()}
assert not (parts["fit"] & parts["tune"] or parts["fit"] & parts["official_valid"] or parts["tune"] & parts["official_valid"])
display(pd.DataFrame({"Partition": list(parts), "Studies": [len(v) for v in parts.values()]}))
audit = json.loads((ROOT / cfg["evaluation"]["directory"] / "leakage_audit.json").read_text())
display(audit)


,Partition,Studies
0,fit,804
1,tune,226
2,official_valid,120
3,calibration,100


{'dataset': 'MRNet-v1.0',
 'split_counts': {'fit': 804,
  'tune': 226,
  'official_valid': 120,
  'calibration': 100},
 'study_overlap': 0,
 'exact_series_overlap_across_splits': 0,
 'within_partition_duplicate_series': 0,
 'patient_grouping_verified': False,
 'limitation': 'Patient linkage is absent from this MRNet copy; study separation cannot prove patient independence.',
 'prior_evaluation_disclosure': 'The official 120-exam validation set was previously evaluated in the earlier baseline; it is not a new external test set.'}

## Interpretation
Only fitting studies determine optimizer updates, feature normalization and class weights. Internal validation determines the architecture, checkpoint and research thresholds. The official validation set was used by an earlier prototype and is not a new external test cohort. There are no RSNA or X-ray cases in these checkpoints.
